<a href="https://colab.research.google.com/github/dylanhogg/jupyter-experiments/blob/fine-tuning/notebooks/finetuning/llama-factory/examples/Finetune_Llama3_with_LLaMA_Factory_v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune Llama-3 with LLaMA Factory v5 (llm-address)

Updated: 2025-10-19 16:47

Github source: https://github.com/dylanhogg/jupyter-experiments/blob/fine-tuning/notebooks/finetuning/llama-factory/examples/Finetune_Llama3_with_LLaMA_Factory_v5.ipynb

LLaMA Factory Project homepage: https://github.com/hiyouga/LLaMA-Factory

Training data: https://huggingface.co/datasets/dylanhogg/gnaf-2022-structured-training-1000000-v0-instruct

v0.2 models:

https://huggingface.co/dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453

### TODO

- enable wandb logging

## Imports

In [1]:
import os
import hashlib
import json
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import files, userdata

## HF Auth

HF auth can be required for gated models you need to be granted access to.

In [2]:
try:
  hf_token = userdata.get("HF_TOKEN")
  print("Loading HF_TOKEN from Colab Secrets...")
  os.environ["HF_TOKEN"] = hf_token
except Exception as e:
  print("No HF_TOKEN Colab Secret set in environment, fall back to interactive login...")
  !hf auth login

Loading HF_TOKEN from Colab Secrets...


## Variables

In [3]:
# https://huggingface.co/unsloth/models?sort=downloads&search=llama
# model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
# model_name_or_path="unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
# model_name_or_path="unsloth/Llama-3.2-1B-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model

# https://huggingface.co/meta-llama/models?sort=downloads&search=llama
# model_name_or_path="meta-llama/Llama-3.2-3B-Instruct"
model_name_or_path="meta-llama/Llama-3.2-1B-Instruct"

## Install Dependencies

In [4]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]  # TODO: swap out for uv

/content
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 416, done.
remote: Counting objects: 100% (416/416), done.
remote: Compressing objects: 100% (317/317), done.
remote: Total 416 (delta 95), reused 321 (delta 81), pack-reused 0 (from 0)
Receiving objects: 100% (416/416), 5.00 MiB | 11.39 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/LLaMA-Factory
assets/       docker/    Makefile        README.md         scripts/  tests/
CITATION.cff  examples/  MANIFEST.in     README_zh.md      setup.py  tests_v1/
data/         LICENSE    pyproject.toml  requirements.txt  src/
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.4.dev0-0.editable-py3-none-a

In [5]:
# Additional packages
!pip install loguru

  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
Using cached loguru-0.7.3-py3-none-any.whl (61 kB)


### Check GPU environment

In [6]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory")

## Fine-tune model via Command Line

Can take awhile...

In [7]:
# Local backup of original dataset_info.json
! [ ! -f data/dataset_info_original.json ] && cp data/dataset_info.json data/dataset_info_original.json

In [8]:
# Create new custom dataset_info.json
# Ref: https://github.com/hiyouga/LLaMA-Factory/blob/main/data/dataset_info.json
dataset_info = {
    "gnaf-2022-structured-training-1000000-v0-instruct-train": {
        "hf_hub_url": "dylanhogg/gnaf-2022-structured-training-1000000-v0-instruct",
        "split": "train",
        "columns": {
          "prompt": "instruction",
          "query": "input",
          "response": "output"
        },
    },
    "gnaf-2022-structured-training-1000000-v0-instruct-test": {
        "hf_hub_url": "dylanhogg/gnaf-2022-structured-training-1000000-v0-instruct",
        "split": "test",
        "columns": {
          "prompt": "instruction",
          "query": "input",
          "response": "output"
        },
    }
}
json.dump(dataset_info, open("data/dataset_info.json", "w", encoding="utf-8"), indent=2)

In [9]:
!cat data/dataset_info.json

{
  "gnaf-2022-structured-training-1000000-v0-instruct-train": {
    "hf_hub_url": "dylanhogg/gnaf-2022-structured-training-1000000-v0-instruct",
    "split": "train",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  },
  "gnaf-2022-structured-training-1000000-v0-instruct-test": {
    "hf_hub_url": "dylanhogg/gnaf-2022-structured-training-1000000-v0-instruct",
    "split": "test",
    "columns": {
      "prompt": "instruction",
      "query": "input",
      "response": "output"
    }
  }
}

In [10]:
# Training set up

# Examples: https://github.com/hiyouga/LLaMA-Factory/tree/main/examples/train_lora

model_version = "v0.2"
aest_now = datetime.now(ZoneInfo("Australia/Sydney")).strftime('%Y%m%d-%H%M%S')

num_train_epochs=2.0
max_samples = 10000
dataset = "gnaf-2022-structured-training-1000000-v0-instruct-train"  # see custom dataset_info.json rendered above
eval_dataset = "gnaf-2022-structured-training-1000000-v0-instruct-test"  # see custom dataset_info.json rendered above

template = "llama3"  # use llama3 prompt template
finetuning_type = "lora"  # use LoRA adapters to save memory
output_dir = "finetune_output"

train_args = dict(
  stage="sft",                                               # do supervised fine-tuning
  do_train=True,
  model_name_or_path=model_name_or_path,
  dataset=dataset,
  eval_dataset=eval_dataset,
  template=template,
  finetuning_type=finetuning_type,
  lora_target="all",                                         # attach LoRA adapters to all linear layers
  output_dir=output_dir,                                     # the path to save LoRA adapters
  plot_loss=True,
  per_device_train_batch_size=2,                             # the micro batch size
  gradient_accumulation_steps=4,                             # the gradient accumulation steps
  lr_scheduler_type="cosine",                                # use cosine learning rate scheduler
  logging_steps=5,                                           # log every 5 steps
  warmup_ratio=0.1,                                          # use warmup scheduler
  save_steps=5000,                                           # save checkpoint every 1000 steps
  learning_rate=5e-5,                                        # the learning rate
  num_train_epochs=num_train_epochs,
  max_samples=max_samples,                                   # data arg: For debugging purposes, truncate the number of examples for each dataset.
  max_grad_norm=1.0,                                         # clip gradient norm to 1.0
  loraplus_lr_ratio=16.0,                                    # use LoRA+ algorithm with lambda=16.0
  fp16=True,                                                 # use float16 mixed precision training
  report_to="none",                                          # disable wandb logging
)

train_args_hash_full = hashlib.sha256(str(train_args).encode("utf-8")).hexdigest()
train_args_hash = train_args_hash_full[0:7]
target_model_name = f"dylanhogg/gnaf-structured-address-{model_version}-{train_args_hash}-{aest_now}"
print(f"{target_model_name=}")
print(f"{train_args_hash=}")
print(f"{train_args=}")

json.dump(train_args, open("train_args.json", "w", encoding="utf-8"), indent=2)

!cat train_args.json

target_model_name='dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453'
train_args={'stage': 'sft', 'do_train': True, 'model_name_or_path': 'meta-llama/Llama-3.2-1B-Instruct', 'dataset': 'gnaf-2022-structured-training-1000000-v0-instruct-train', 'eval_dataset': 'gnaf-2022-structured-training-1000000-v0-instruct-test', 'template': 'llama3', 'finetuning_type': 'lora', 'lora_target': 'all', 'output_dir': 'finetune_output', 'plot_loss': True, 'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 4, 'lr_scheduler_type': 'cosine', 'logging_steps': 5, 'warmup_ratio': 0.1, 'save_steps': 1000, 'learning_rate': 5e-05, 'num_train_epochs': 2.0, 'max_samples': 10000, 'max_grad_norm': 1.0, 'loraplus_lr_ratio': 16.0, 'fp16': True, 'report_to': 'none'}
train_args_hash='712c28b'
{
  "stage": "sft",
  "do_train": true,
  "model_name_or_path": "meta-llama/Llama-3.2-1B-Instruct",
  "dataset": "gnaf-2022-structured-training-1000000-v0-instruct-train",
  "eval_dataset": "gnaf-2022-stru

In [11]:
# Do training

%cd /content/LLaMA-Factory/
print(f"{target_model_name=}")
print(f"{train_args=}")
print(f"{train_args_hash=}")

!cat train_args.json
!llamafactory-cli train train_args.json

/content/LLaMA-Factory
target_model_name='dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453'
train_args={'stage': 'sft', 'do_train': True, 'model_name_or_path': 'meta-llama/Llama-3.2-1B-Instruct', 'dataset': 'gnaf-2022-structured-training-1000000-v0-instruct-train', 'eval_dataset': 'gnaf-2022-structured-training-1000000-v0-instruct-test', 'template': 'llama3', 'finetuning_type': 'lora', 'lora_target': 'all', 'output_dir': 'finetune_output', 'plot_loss': True, 'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 4, 'lr_scheduler_type': 'cosine', 'logging_steps': 5, 'warmup_ratio': 0.1, 'save_steps': 1000, 'learning_rate': 5e-05, 'num_train_epochs': 2.0, 'max_samples': 10000, 'max_grad_norm': 1.0, 'loraplus_lr_ratio': 16.0, 'fp16': True, 'report_to': 'none'}
train_args_hash='712c28b'
{
  "stage": "sft",
  "do_train": true,
  "model_name_or_path": "meta-llama/Llama-3.2-1B-Instruct",
  "dataset": "gnaf-2022-structured-training-1000000-v0-instruct-train",
  "eval_da

In [ ]:
!ls /content/LLaMA-Factory/finetune_output

In [ ]:
!zip -r finetune_output.zip /content/LLaMA-Factory/finetune_output

In [ ]:
!ls -lha /content/LLaMA-Factory/finetune_output.zip

In [ ]:
files.download("/content/LLaMA-Factory/finetune_output.zip")

## Infer the fine-tuned model

In [ ]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

%cd /content/LLaMA-Factory/

args = dict(
  # model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  model_name_or_path=model_name_or_path,
  adapter_name_or_path="finetune_output",                        # load the saved LoRA adapters
  template=template,                                         # same to the one in training
  finetuning_type=finetuning_type,                           # same to the one in training
)
chat_model = ChatModel(args)

def build_user_message(instruction: str, query: str = "") -> str:
    """
    Reconstruct the same input format the model saw in training.
    """
    if query:
        return f"{instruction}\n{query}\n"
    return f"{instruction}\n"

messages = []
print("Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.")
while True:
  query = input("\nUser: ")
  if query.strip() == "exit":
    break
  if query.strip() == "clear":
    messages = []
    torch_gc()
    print("History has been removed.")
    continue

  instruction = "Translate a text address into stuctured json."  # TODO: get from training dataset!
  user_message = build_user_message(instruction, query)
  messages.append({"role": "user", "content": user_message})
  print("Assistant: ", end="", flush=True)

  response = ""
  for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text
  print()
  messages.append({"role": "assistant", "content": response})

torch_gc()

## Merge the LoRA adapter with the base model and upload to HF

NOTE: the Colab free version has merely 12GB RAM, where merging LoRA of a 8B model needs at least 18GB RAM

In [12]:
print(f"{target_model_name=}")

target_model_name='dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453'


In [13]:
# Create model repo on Huggingface as private, ready to push to after model merge
!hf repo create {target_model_name} --repo-type model --private

Successfully created dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453 on the Hub.
Your repo is now available at https://huggingface.co/dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453


In [14]:
# Merge and export model

args = dict(
  model_name_or_path=model_name_or_path,                    # use official non-quantized Llama-3-8B-Instruct model
  adapter_name_or_path="finetune_output",                   # load the saved LoRA adapters
  template=template,                                        # same to the one in training
  finetuning_type="lora",                                   # same to the one in training

  # export
  export_dir="finetune_output_merged",                      # the path to save the merged model
  export_size=2,                                            # the file shard size (in GB) of the merged model
  export_device="cpu",                                      # the device used in export, can be chosen from `cpu` and `auto`
  export_hub_model_id=target_model_name                     # the Hugging Face hub ID to upload model
)

json.dump(args, open("finetune_output_merged.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/

!llamafactory-cli export finetune_output_merged.json

/content/LLaMA-Factory
2025-10-19 06:39:44.685610: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760855984.708267   13878 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760855984.715509   13878 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760855984.732229   13878 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760855984.732262   13878 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760855984.732266   13878 computation_placer.cc:177]

## Review merged model outputs and update HF model README

In [15]:
!ls -lha /content/LLaMA-Factory/finetune_output

total 39M
drwxr-xr-x  5 root root 4.0K Oct 19 06:39 .
drwxr-xr-x 14 root root 4.0K Oct 19 06:40 ..
-rw-r--r--  1 root root  942 Oct 19 06:39 adapter_config.json
-rw-r--r--  1 root root  22M Oct 19 06:39 adapter_model.safetensors
-rw-r--r--  1 root root  205 Oct 19 06:39 all_results.json
-rw-r--r--  1 root root 3.8K Oct 19 06:39 chat_template.jinja
drwxr-xr-x  2 root root 4.0K Oct 19 06:20 checkpoint-1000
drwxr-xr-x  2 root root 4.0K Oct 19 06:33 checkpoint-2000
drwxr-xr-x  2 root root 4.0K Oct 19 06:39 checkpoint-2500
-rw-r--r--  1 root root 1.5K Oct 19 06:39 README.md
-rw-r--r--  1 root root  506 Oct 19 06:39 special_tokens_map.json
-rw-r--r--  1 root root  50K Oct 19 06:39 tokenizer_config.json
-rw-r--r--  1 root root  17M Oct 19 06:39 tokenizer.json
-rw-r--r--  1 root root  89K Oct 19 06:39 trainer_log.jsonl
-rw-r--r--  1 root root  80K Oct 19 06:39 trainer_state.json
-rw-r--r--  1 root root 6.1K Oct 19 06:39 training_args.bin
-rw-r--r--  1 root root  33K Oct 19 06:39 training_loss.

In [16]:
!ls -lha /content/LLaMA-Factory/finetune_output_merged

total 2.4G
drwxr-xr-x  2 root root 4.0K Oct 19 06:41 .
drwxr-xr-x 14 root root 4.0K Oct 19 06:40 ..
-rw-r--r--  1 root root 3.8K Oct 19 06:41 chat_template.jinja
-rw-r--r--  1 root root  866 Oct 19 06:40 config.json
-rw-r--r--  1 root root  184 Oct 19 06:40 generation_config.json
-rw-r--r--  1 root root 1.9G Oct 19 06:40 model-00001-of-00002.safetensors
-rw-r--r--  1 root root 453M Oct 19 06:40 model-00002-of-00002.safetensors
-rw-r--r--  1 root root  498 Oct 19 06:41 Modelfile
-rw-r--r--  1 root root  12K Oct 19 06:40 model.safetensors.index.json
-rw-r--r--  1 root root  506 Oct 19 06:41 special_tokens_map.json
-rw-r--r--  1 root root  50K Oct 19 06:41 tokenizer_config.json
-rw-r--r--  1 root root  17M Oct 19 06:41 tokenizer.json


In [17]:
# Automatically generated model README.md
# TODO: replace sections in README
!cat /content/LLaMA-Factory/finetune_output/README.md

---
library_name: peft
license: other
base_model: meta-llama/Llama-3.2-1B-Instruct
tags:
- base_model:adapter:meta-llama/Llama-3.2-1B-Instruct
- llama-factory
- lora
- transformers
pipeline_tag: text-generation
model-index:
- name: finetune_output
  results: []
---

<!-- This model card has been generated automatically according to the information the Trainer had access to. You
should probably proofread and complete it, then remove this comment. -->

# finetune_output

This model is a fine-tuned version of [meta-llama/Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) on the gnaf-2022-structured-training-1000000-v0-instruct-train dataset.

## Model description

More information needed

## Intended uses & limitations

More information needed

## Training and evaluation data

More information needed

## Training procedure

### Training hyperparameters

The following hyperparameters were used during training:
- learning_rate: 5e-05
- train_batch_size: 2
- eval

In [18]:
# Append training info to autogenerated README
print(f"{model_version=}")
print(f"{aest_now=}")
print(f"{target_model_name=}")
print(f"{train_args=}")
print(f"{train_args_hash=}")

# Local backup of original /finetune_output/README.md
! [ ! -f /content/LLaMA-Factory/finetune_output/README_original.md ] && cp /content/LLaMA-Factory/finetune_output/README.md /content/LLaMA-Factory/finetune_output/README_original.md

with open("/content/LLaMA-Factory/finetune_output/README.md", "a") as f:
  f.write("\n### Full training details\n\n")
  f.write(f"{model_version=}\n")
  f.write(f"{aest_now=}\n")
  f.write(f"{target_model_name=}\n")
  f.write(f"{train_args_hash=}\n")
  f.write("train_args=\n")
  f.write(f"{json.dumps(train_args, indent=2)}\n")
  f.write("\n")

model_version='v0.2'
aest_now='20251019-170453'
target_model_name='dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453'
train_args={'stage': 'sft', 'do_train': True, 'model_name_or_path': 'meta-llama/Llama-3.2-1B-Instruct', 'dataset': 'gnaf-2022-structured-training-1000000-v0-instruct-train', 'eval_dataset': 'gnaf-2022-structured-training-1000000-v0-instruct-test', 'template': 'llama3', 'finetuning_type': 'lora', 'lora_target': 'all', 'output_dir': 'finetune_output', 'plot_loss': True, 'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 4, 'lr_scheduler_type': 'cosine', 'logging_steps': 5, 'warmup_ratio': 0.1, 'save_steps': 1000, 'learning_rate': 5e-05, 'num_train_epochs': 2.0, 'max_samples': 10000, 'max_grad_norm': 1.0, 'loraplus_lr_ratio': 16.0, 'fp16': True, 'report_to': 'none'}
train_args_hash='712c28b'


In [19]:
# Upload model README to HF
!hf upload {target_model_name} /content/LLaMA-Factory/finetune_output/README.md /README.md

https://huggingface.co/dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453/blob/main//README.md


In [20]:
# Upload additional supporting files
# TODO: put in a subfolder
# TODO: "train_args.json"
# TODO: "finetune_output_merged.json"
# TODO: finetune_output/training_loss.png
# TODO: loguru training logs

# Upload additional supporting files to HF
!hf upload {target_model_name} /content/LLaMA-Factory/train_args.json /train_args.json
!hf upload {target_model_name} /content/LLaMA-Factory/finetune_output_merged.json /finetune_output_merged.json
!hf upload {target_model_name} /content/LLaMA-Factory/finetune_output/training_loss.png /training_loss.png

https://huggingface.co/dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453/blob/main//train_args.json
https://huggingface.co/dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453/blob/main//finetune_output_merged.json
https://huggingface.co/dylanhogg/gnaf-structured-address-v0.2-712c28b-20251019-170453/blob/main//training_loss.png


## Zip merged model for download

In [ ]:
# NOTE: can be slow since it's merged in the large base model
!zip -r finetune_output_merged.zip /content/LLaMA-Factory/finetune_output_merged

In [ ]:
!ls -lha /content/LLaMA-Factory/finetune_output_merged.zip

In [ ]:
files.download("/content/LLaMA-Factory/finetune_output_merged.zip")

In [21]:
print("Done!")

Done!
